# 02 — Dataset exploration and validation

        **Estimated time:** 45 minutes<br>
        **Prerequisites:** 01 — Dataset provenance and license<br>
        **Learner-produced evidence:** a privacy-preserving quality report and dataset-card draft

        ## Learning objectives

        - Measure missing, duplicate, label, length, and sensitive-pattern findings.
- Use the exact local tokenizer for token-length analysis.
- Interpret aggregate evidence without displaying unnecessary raw text.

        This notebook is a teaching interface over the reusable code in `src/`.
        It uses only prepared local files. Run `make prepare-flight` before the
        trip; no cell installs packages or downloads data.


In [ ]:
from aai_local_finetuning.offline import enable_offline_environment

enable_offline_environment()

## Load the inspected local source

Reusable audit logic lives under `src/aai_local_finetuning/data`.
The notebook asks questions of the resulting evidence rather than
reimplementing the pipeline in cells.


In [ ]:
import json

from aai_local_finetuning.data import (
    audit_dataset,
    check_split_files,
    summarize_instruction_tokens,
)
from aai_local_finetuning.settings import load_settings

settings = load_settings()
raw_csv = settings.csv_path
processed_dir = settings.processed_dir
if not raw_csv.is_file():
    raise FileNotFoundError(
        "The immutable Bitext CSV is missing. Prepare this machine online."
    )

## Quality audit

The report contains counts and distributions, not raw samples. Exact
duplicates are measured after canonicalization. Near duplicates and
inferred templates are heuristic evidence and must be documented as such.


In [ ]:
audit = audit_dataset(raw_csv)
audit_payload = audit.model_dump(mode="json")
core_quality = {
    key: audit_payload[key]
    for key in (
        "source_records",
        "valid_records",
        "unique_records",
        "curated_records",
        "invalid_record_count",
        "missing_by_field",
        "exact_duplicate_count",
        "exact_duplicate_rate",
        "near_duplicate_pairs",
        "near_duplicate_clusters",
        "conflicting_group_count",
        "excluded_conflicting_records",
    )
}
core_quality

## Labels, lengths, language, and sensitive-looking patterns

Pattern matches are counts only. Email-, URL-, phone-like text and
placeholders are masked before portable training records are written.
Source flags remain explicit evaluation slices; difficulty is a separate
versioned heuristic, not a human quality label.


In [ ]:
token_lengths = summarize_instruction_tokens(raw_csv, settings.model_dir)
distribution_report = {
    "intents": audit_payload["intent_distribution"],
    "categories": audit_payload["category_distribution"],
    "languages": {settings.dataset.language: audit_payload["source_records"]},
    "instruction_characters": audit_payload["instruction_characters"],
    "instruction_word_proxy": audit_payload["instruction_words"],
    "pinned_tokenizer_tokens": token_lengths.model_dump(mode="json"),
    "source_flags": audit_payload["flag_distribution"],
    "difficulty": audit_payload["difficulty_distribution"],
    "sensitive_pattern_counts": audit_payload["sensitive_pattern_counts"],
}
distribution_report

## Automated split-integrity gate

This gate examines the prepared evidence boundaries without displaying
frozen test content. It checks exact, inferred-template, and near-duplicate
overlap plus target and demonstration leakage.


In [ ]:
integrity = check_split_files(processed_dir)
integrity.model_dump(mode="json")

## Dataset-card draft inputs

The tracked card remains reviewed prose. This draft makes measurements
easy to revisit whenever source bytes, processing, or split policy change.


In [ ]:
manifest = json.loads((processed_dir / "manifest.json").read_text(encoding="utf-8"))
dataset_card_draft = {
    "source": manifest["dataset"],
    "fingerprint": manifest["dataset_fingerprint"],
    "quality": core_quality,
    "label_count": len(audit_payload["intent_distribution"]),
    "sensitive_review": audit_payload["sensitive_pattern_counts"],
    "split_strategy": manifest["split_strategy"],
    "review_required": [
        "near-duplicate threshold",
        "sensitive-looking content",
        "generated response policy",
        "redistribution obligations",
    ],
}
dataset_card_draft

## Exercise — identify the highest-risk assumption

Pick one measured finding and state what could go wrong if it were
ignored. Success means your answer connects a finding to either leakage,
fairness across labels, privacy, or misleading evaluation.


In [ ]:
finding = "near-duplicate clusters"
risk = (
    "Related templates crossing splits could make memorization look like "
    "generalization, so groups must stay inside one evidence boundary."
)
assert finding and len(risk.split()) >= 10
{"finding": finding, "risk": risk}

**Hint:** counts are not conclusions. Ask how each observation could bias
the final comparison or expose content unnecessarily.


## Checkpoint

The source has now been measured, not merely described. Keep the frozen
test content out of prompt design and model development.

**Next:** `03_leakage_safe_splits.ipynb` follows records from immutable
source to portable train, validation, and test boundaries.
